# 文件概览

在 `deephall` 文件夹中，各个文件的功能和依赖关系如下：

### 1. `__init__.py`
这个文件是包的初始化文件，用于导出 `deephall` 包中最常用的类和函数，如 `Config` 和 `train`。

### 2. `config.py`
定义了所有配置相关的类，如 `Config`, `System`, `Network`, `MCMC`, `Optimizer` 等。这些类用于设置和管理模拟训练的各种参数。

### 3. `constants.py`
包含了一些全局常量和函数，如 `pmap` 和 `pmean`，这些常用于并行处理和数据同步。

### 4. `hamiltonian.py`
定义了与哈密顿量相关的函数，如局部动能和势能的计算。这些函数直接与物理模型的核心计算相关。

### 5. `log.py`
提供了日志管理和检查点管理的功能，如 `LogManager` 和 `StatsWriter` 类，用于处理训练过程中的日志记录和状态保存。

### 6. `loss.py`
定义了损失函数的计算，这是训练神经网络时用于优化的关键函数。

### 7. `mcmc.py`
包含了用于蒙特卡洛马尔可夫链（MCMC）步骤的函数，这是物理模拟中用于样本生成的关键技术。

### 8. `train.py`
包含了整个训练流程的实现，如初始化、训练循环和命令行接口。这是整个包的核心，调用了其他模块如 `log`, `loss`, `mcmc` 等来执行训练任务。

### 9. `types.py`
定义了一些用于类型注解的类和接口，如 `LogPsiNetwork` 和 `TrainingStep`。这些类型帮助确保代码的清晰和正确性。

### 文件依赖关系：
- `train.py` 依赖于 `log.py`, `loss.py`, `mcmc.py`, `types.py`, `config.py` 和 `constants.py`，因为它需要调用这些模块提供的函数和类来执行训练。
- `loss.py` 和 `mcmc.py` 可能会依赖于 `hamiltonian.py` 来获取能量计算函数。
- `log.py` 依赖于 `config.py` 来获取配置信息，以及 `types.py` 来使用定义的数据类型。
- `config.py` 是基础配置模块，通常不依赖于其他模块。
- `constants.py` 提供了基础的工具函数，可能被多个模块依赖。

这些文件共同构成了一个完整的物理模拟训练框架，通过相互协作实现了从配置管理到训练执行的全过程。


# Train.py

`train.py` 是 DeepHall 项目的核心训练脚本，负责启动和管理量子霍尔效应的模拟训练过程。以下是代码的工作顺序和主要逻辑：

### 1. **初始化阶段**
   - **日志初始化**：`init_logging()` 初始化日志系统，确保训练过程中的信息能够被记录。
   - **配置解析**：通过 `cli()` 函数解析命令行参数，加载配置文件（YML 或命令行键值对），并生成最终的配置对象 `Config`。
   - **日志管理器**：`LogManager(cfg)` 创建日志管理器，用于管理训练过程中的日志记录和检查点保存。

### 2. **训练准备阶段**
   - **模型初始化**：`make_network(cfg.system, cfg.network)` 根据配置创建神经网络模型。
   - **初始状态生成**：`initialize_state(cfg, model)` 生成初始的电子坐标和模型参数，并将这些数据分布到所有设备上。
   - **MCMC 设置**：`setup_mcmc(cfg, network)` 配置 MCMC（马尔可夫链蒙特卡洛）步骤，用于在训练过程中生成样本。
   - **优化器设置**：`optimizers.make_optimizer_step(cfg, network)` 创建优化器初始化函数和训练步骤。

### 3. **训练循环阶段**
   - **MCMC 预热**：如果从初始步数开始，会进行 MCMC 预热（`burn_in`），以确保 MCMC 链达到稳定状态。
   - **初始能量计算**：如果需要，计算并记录初始能量，用于调试和验证。
   - **主训练循环**：进入 `train_loop(cfg, log_manager)`，逐次执行以下步骤：
     - **MCMC 采样**：通过 `pmap_mcmc_step` 生成新的样本数据。
     - **MCMC 步长更新**：根据接受率动态调整 MCMC 步长。
     - **优化器更新**：通过 `training_step` 更新模型参数，计算损失和统计信息。
     - **日志记录**：将训练步数、能量、方差等统计信息记录到日志中。
     - **检查点保存**：根据配置的时间间隔或步数间隔保存检查点，确保训练过程可以从中断处恢复。

### 4. **终止处理**
   - **优雅退出**：如果收到 `SIGINT` 或 `SIGTERM` 信号，`GracefulKiller` 会标记需要终止，并在退出前保存检查点。
   - **异常终止**：如果能量值出现 `NaN`，训练会立即终止，并抛出异常。

### 5. **训练结束**
   - **时间统计**：训练结束后，计算并记录每步的平均时间，用于性能分析。

### 6. **命令行接口**
   - **`cli()` 函数**：作为程序的入口，负责解析命令行参数，加载配置，并启动训练过程。支持以下功能：
     - 通过 `dotlist` 参数传递配置键值对。
     - 通过 `--yml` 参数加载 YML 配置文件。
     - 通过 `--debug` 参数禁用 JAX 的 `pmap` 和 `jit`，方便调试。

### 总结
`train.py` 的工作顺序可以概括为：
1. **初始化**：配置、日志、模型、初始状态。
2. **训练准备**：MCMC、优化器设置。
3. **训练循环**：MCMC 采样、优化器更新、日志记录、检查点保存。
4. **终止处理**：优雅退出或异常终止。
5. **训练结束**：时间统计和日志记录。

整个脚本通过模块化的设计，将训练过程的各个部分解耦，便于维护和扩展。


# hamiltonian.py